In [40]:
from tespy.networks import Network

In [41]:
my_plant = Network()

In [42]:
my_plant.units.set_defaults(
    temperature="K",
    pressure="bar",
    pressure_difference="bar",
    enthalpy="J/kg",
    heat="W",
    power="W",
)

Setting up components

In [43]:
from tespy.components import (
    CycleCloser,Compressor,Valve,SimpleHeatExchanger
)

In [44]:
cycle_closer = CycleCloser("cycle closer")

In [45]:
#heat sink
condenser = SimpleHeatExchanger('condenser')

In [46]:
#heat source
evaporator = SimpleHeatExchanger('evaporator')

In [47]:
valve = Valve("expansion valve")
compressor = Compressor("compressor")

Making connections

Connections are used to link two components (outlet of component 1 to inlet of component 2: source to target). If two components are connected with each other the fluid properties at the source will be equal to the properties at the target. It is possible to set the properties on each connection similarly as parameters are set for components. The basic specification options are:

    mass flow (m)

    volumetric flow (v)

    pressure (p)

    enthalpy (h)

    temperature (T)

    a fluid vector (fluid)


In [48]:
from tespy.connections import Connection


In [49]:
#c1 = Connection(source, source_id,target,target_id)
c1 = Connection(cycle_closer,'out1', evaporator, 'in1', label='1')
c2 = Connection(evaporator,'out1', compressor, 'in1', label='2')
c3 = Connection(compressor,'out1', condenser, 'in1', label='3')
c4 = Connection(condenser,'out1', valve, 'in1', label='4')
c0 = Connection(valve,'out1', cycle_closer, 'in1', label='0')


In [50]:
my_plant.add_conns(c0,c1,c2,c3,c4)

In [51]:
condenser.set_attr(pr=0.98, Q=-1000)
evaporator.set_attr(pr=0.98)
compressor.set_attr(eta_s=0.85)

c2.set_attr(T=(20+273.15), x=1, fluid={'R134a': 1})
c4.set_attr(T=(273.15+ 80), x=0)


In [52]:
my_plant.solve(mode='design')





 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 4.03e+05   | 4 %        | 8.25e-01   | 0.00e+00   | 1.18e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 9.72e+04   | 11 %       | 7.83e-01   | 0.00e+00   | 3.85e-11   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 1.81e-08   | 100 %      | 1.46e-13   | 0.00e+00   | 3.85e-11   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 3.27e-11   | 100 %      | 2.08e-18   | 0.00e+00   | 3.85e-11   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 4, Calculation time: 0.02 s, Iterations per second: 254.61


In [53]:
my_plant.print_results()

print(f'COP = {abs(condenser.Q.val) / compressor.P.val}')


##### RESULTS (Compressor) #####
+------------+----------+----------+-----------+----------+
|            |        P |       pr |        dp |    eta_s |
|------------+----------+----------+-----------+----------|
| compressor | 2.96e+02 | 4.70e+00 | -2.12e+01 | 8.50e-01 |
+------------+----------+----------+-----------+----------+
##### RESULTS (SimpleHeatExchanger) #####
+------------+-----------+----------+----------+-----------+----------+
|            |         Q |       pr |       dp |   zeta_d4 |     zeta |
|------------+-----------+----------+----------+-----------+----------|
| condenser  | -1.00e+03 | 9.80e-01 | 5.37e-01 |  2.44e+11 | 2.44e+11 |
| evaporator |  7.04e+02 | 9.80e-01 | 1.17e-01 |  8.11e+09 | 8.11e+09 |
+------------+-----------+----------+----------+-----------+----------+
##### RESULTS (CycleCloser) #####
+--------------+------------------+-------------------+
|              |   mass_deviation |   fluid_deviation |
|--------------+------------------+-----------